# 12. Hyperparameter Tuning and Model Optimization

This notebook performs Bayesian hyperparameter optimization on the best-performing model (XGBoost) identified from the model comparison analysis. The goal is to fine-tune the model to achieve optimal performance on the student dropout prediction task.

## Objectives
- Load the feature-selected dataset and baseline models from previous steps
- Configure Bayesian hyperparameter search space for XGBoost
- Perform efficient hyperparameter optimization using BayesSearchCV
- Evaluate tuned model performance through cross-validation
- Test generalization capability on the held-out test set
- Save the optimized model and results for deployment

## Methods
- **Bayesian Optimization** - Efficient hyperparameter search using scikit-optimize
- **Cross-Validation** - 10-fold repeated stratified cross-validation for robust evaluation
- **PR-AUC Optimization** - Primary metric for model selection in imbalanced classification
- **Generalization Testing** - Final evaluation on unseen test data

## Table of Contents
1. **[Setup & Data Loading](#setup--data-loading)** - Import libraries and load processed data and baseline models
2. **[Hyperparameter Search Space Configuration](#2-hyperparameter-search-space-configuration)** - Define search spaces and prepare data for optimization
3. **[Hyperparameter Optimization](#hyperparameter-optimization)** - Run Bayesian optimization with cross-validation
4. **[Generalization](#4-generalization)** - Evaluate final tuned model performance and generalization
5. **[Save tuned model + tuning outputs](#5-save-tuned-model--tuning-outputs)** - Persist optimized model and results

---

## 1. Setup & Data Loading {#setup--data-loading}

Import required libraries and load the feature-selected dataset and baseline models from previous steps.

In [ ]:
# Core
import os
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.base import clone
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

# Bayesian optimization (scikit-optimize)
from skopt import BayesSearchCV
from skopt.space import Real, Integer

# Model
from xgboost import XGBClassifier

### 1.2 Load Processed Data

In [ ]:
# Define data directory
current_user = os.getenv('USER')
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")

# Load data and models
df = pd.read_csv(PROCESSED_DIR / 'feature_selection.csv')

# Load 'model_comparison_analysis.pkl'
model_data = joblib.load(PROCESSED_DIR / 'model_comparison_analysis.pkl')

### 1.3 Rebuild splits from set and define target

In [ ]:
# Purpose: Define the target column and reconstruct the train/validation/test splits
#          using the precomputed 'set' labels to avoid leakage and keep tuning consistent.

target_col = 'sdo5_degree_drop_out'

# Split data based on the 'set' column
train_df = df[df['set'] == 'train'].copy()
val_df   = df[df['set'] == 'val'].copy()
test_df  = df[df['set'] == 'test'].copy()

print(f"Training set shape: {train_df.shape}")
print(f"Validation set shape: {val_df.shape}")
print(f"Test set shape: {test_df.shape}")


### 1.4 Model Selection {#model-selection}

Based on the model comparison results, select the top performing models for hyperparameter tuning.

In [ ]:
# Purpose: Load saved model-comparison outputs and select only the previously chosen best model (XGBoost)
#          to continue with hyperparameter tuning in a controlled and reproducible way.

cv_results = model_data['cv_results']

models_to_tune = {"XGBoost": model_data['models']["XGBoost"]}

print("Model selected for hyperparameter tuning:")
print("- XGBoost")

In [ ]:
# Remove all sdo78_ columns
sdo78_cols = [col for col in df.columns if col.startswith('sdo78_')]
df = df.drop(columns=sdo78_cols)

## 2. Hyperparameter Search Space Configuration 

### 2.1 Define hyperparameter search space for XGB

In [ ]:
# Purpose: Define the Bayesian hyperparameter search space for XGBoost.
#          The search space is aligned with the baseline configuration used in the
#          model comparison notebook, preserving the same set of performance-relevant
#          hyperparameters (depth, learning rate, trees, sampling, and regularization).
#          Non-performance parameters (e.g. random_state, verbosity, eval_metric, n_jobs)
#          are intentionally kept fixed and excluded from tuning for reproducibility
#          and interpretability.

xgb_search_space = {
    'n_estimators': Integer(50, 400, name='n_estimators'),
    'max_depth': Integer(3, 8, name='max_depth'),
    'learning_rate': Real(0.01, 0.2, prior='log-uniform', name='learning_rate'),
    'subsample': Real(0.6, 1.0, prior='uniform', name='subsample'),
    'colsample_bytree': Real(0.6, 1.0, prior='uniform', name='colsample_bytree'),
    'reg_alpha': Real(1e-4, 1.0, prior='log-uniform', name='reg_alpha'),
    'reg_lambda': Real(0.5, 5.0, prior='log-uniform', name='reg_lambda'),
    'min_child_weight': Integer(1, 10, name='min_child_weight')
}
print("XGBoost hyperparameter search space defined.")

### 2.2 Build X_train_val, y_train_val, compute SPW, create tuned base estimator

In [ ]:
# Purpose: Reconstruct X_train_val and y_train_val from the existing train/val splits,
#          compute a fixed scale_pos_weight on train+val only, and create a fresh XGB estimator
#          that will be tuned with consistent imbalance handling, ie 
#          evaluating all hyperparameter configurations under the same class-imbalance assumption.

# Define feature columns (exclude split label + target)
exclude_cols = ['set', target_col]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Build train+val matrices for tuning
train_val_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_train_val = train_val_df[feature_cols].copy()
y_train_val = train_val_df[target_col].copy()

# Compute and fix scale_pos_weight using train+val only
neg = (y_train_val == 0).sum()
pos = (y_train_val == 1).sum()
spw = (neg / pos) if pos > 0 else 1.0

xgb_base = clone(models_to_tune["XGBoost"]).set_params(scale_pos_weight=spw)

print(f"Number of features: {len(feature_cols)}")
print(f"Train+Val shape: {X_train_val.shape}")
print(f"Fixed scale_pos_weight: {spw:.3f}")

## 3. Hyperparameter Optimization {#hyperparameter-optimization}

### 3.1 Define CV strategy and scoring (PR-AUC is the refit metric)

In [ ]:
# Purpose: Define a stable cross-validation strategy (RepeatedStratifiedKFold) and
#          a multi-metric scoring dictionary where PR-AUC is the primary optimization target.
#          Other metrics are tracked for reporting and sanity checks.

N_FOLDS = 10
N_REPEATS = 5

cv_strategy = RepeatedStratifiedKFold(
    n_splits=N_FOLDS,
    n_repeats=N_REPEATS,
    random_state=42
)

scoring = {
    'pr_auc': 'average_precision',
    'roc_auc': 'roc_auc',
    'accuracy': 'accuracy',
    'f1': 'f1',
    'recall': 'recall',
    'precision': 'precision'
}

print(f"CV strategy: {N_FOLDS}-fold × {N_REPEATS} repeats = {N_FOLDS * N_REPEATS} evaluations per trial")
print(f"Scoring metrics: {list(scoring.keys())}")
print("Refit metric: pr_auc")

### 3.2 Run Bayesian optimization (BayesSearchCV)

Why Bayesian optimization :

XGBoost has continuous and interacting hyperparameters; Bayesian optimization explores these far more efficiently than GridSearch or RandomSearch.

Each evaluation is expensive (repeated stratified CV); Bayesian optimization minimizes wasted trials.

PR-AUC is noisy and harder to optimize; Bayesian methods handle this better than brute-force searches.

 The goal was fine-tuning, not broad exploration—Bayesian optimization is ideal for local refinement.

It aligns with the existing careful CV, imbalance handling, and statistical evaluation setup.

In [ ]:
# Purpose: Run Bayesian hyperparameter optimization for XGBoost.
#          The search selects the best configuration by PR-AUC (Average Precision),
#          while storing all other metrics for later comparison and reporting.

N_ITER = 30  # number of times the hyperparameter space is explored, with 10x5 CV setup.

bayes_search = BayesSearchCV(
    estimator=xgb_base,            # already has fixed scale_pos_weight
    search_spaces=xgb_search_space,
    n_iter=N_ITER,
    scoring=scoring,
    refit='pr_auc',
    cv=cv_strategy,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

bayes_search.fit(X_train_val, y_train_val)

print("Bayesian optimization complete.")
print(f"Best PR-AUC (CV mean): {bayes_search.best_score_:.4f}")
print("Best hyperparameters:")
print(bayes_search.best_params_)

#### Results: 

Baseline vs tuned performance:

Baseline PR-AUC (model comparison): 0.63

PR-AUC after Bayesian tuning with 42 seeds: 0.64

PR-AUC after Bayesian tuning with 0 seed: 0.64

PR-AUC after Bayesian tuning with 2026 seeds: 0.64

#### Why this improvement matters:

The evaluation setup is strong (stratified CV with repeats).

PR-AUC is difficult to improve, especially in imbalanced settings.

The baseline XGBoost was already well configured.

#### What Bayesian optimization confirmed:

The tuning refined rather than overhauled the model.

Gains came from more trees and slower learning, not increased complexity.

Tree depth remained unchanged, supporting model stability.

#### Key takeaway:

Bayesian optimization polished an already strong XGBoost model, yielding a meaningful improvement in precision–recall performance for dropout prediction.

In [ ]:
# # Purpose: Automatically detect an "elbow" (diminishing returns point) in Bayesian optimization
# #          by finding the iteration with maximum deviation from the straight line connecting
# #          the first and last points of the best-so-far PR-AUC curve
# #          (triangle / max-distance method).


# # ---------------------------------------------------------
# # 1) Extract per-iteration PR-AUC and compute best-so-far
# # ---------------------------------------------------------
# results = pd.DataFrame(bayes_search.cv_results_)

# pr_auc_scores = results["mean_test_pr_auc"].to_numpy()
# best_so_far = np.maximum.accumulate(pr_auc_scores)
# iters = np.arange(1, len(best_so_far) + 1)

# # ---------------------------------------------------------
# # 2) Elbow detection: max distance to line between endpoints
# # ---------------------------------------------------------
# x1, y1 = iters[0], best_so_far[0]
# x2, y2 = iters[-1], best_so_far[-1]

# den = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)

# if den == 0:
#     elbow_iter = int(iters[-1])
# else:
#     num = np.abs(
#         (y2 - y1) * iters
#         - (x2 - x1) * best_so_far
#         + (x2 * y1 - y2 * x1)
#     )
#     dists = num / den
#     elbow_idx = int(np.argmax(dists))
#     elbow_iter = int(iters[elbow_idx])

# print(f"Estimated elbow iteration (diminishing returns): {elbow_iter}")

# # ---------------------------------------------------------
# # 3) Plot convergence with elbow marker (readable formatting)
# # ---------------------------------------------------------
# plt.figure(figsize=(12, 6))

# plt.plot(
#     iters,
#     best_so_far,
#     marker="o",
#     linewidth=2.5,
#     markersize=6,
#     label="Best PR-AUC so far"
# )

# plt.axvline(
#     elbow_iter,
#     linestyle="--",
#     linewidth=2.5,
#     color="tab:red",
#     label=f"Elbow ≈ iteration {elbow_iter}"
# )

# # Axis labels
# plt.xlabel("Bayesian Optimization Iteration", fontsize=14)
# plt.ylabel("Best PR-AUC so far", fontsize=14)

# # Title
# plt.title(
#     "Bayesian Optimization Convergence (PR-AUC) with Elbow Detection, 42 Seeds",
#     fontsize=16,
#     fontweight="bold",
#     pad=14
# )

# # Increase tick label font sizes
# plt.xticks(fontsize=12)
# plt.yticks(fontsize=12)

# # Grid and legend
# plt.grid(alpha=0.3)
# plt.legend(fontsize=12)

# plt.tight_layout()
# plt.show()


In [ ]:
# # Purpose: Add a simple, transparent stopping check for Bayesian optimization.
# #          If the best-so-far PR-AUC does NOT improve by at least 0.001 over the last 10 iterations,
# #          we flag that the search has likely plateaued (typical stopping criterion).


# # --- Extract PR-AUC per iteration (in the order evaluated) ---
# results = pd.DataFrame(bayes_search.cv_results_)
# pr_auc_scores = results["mean_test_pr_auc"].to_numpy()

# # --- Best-so-far curve ---
# best_so_far = np.maximum.accumulate(pr_auc_scores)

# # --- Typical stopping criterion parameters ---
# WINDOW = 10
# MIN_GAIN = 0.001

# if len(best_so_far) < WINDOW + 1:
#     print(f"Stopping check skipped: need at least {WINDOW + 1} iterations, got {len(best_so_far)}.")
# else:
#     # Gain over the last WINDOW iterations (compare best now vs best WINDOW iterations ago)
#     gain_last_window = best_so_far[-1] - best_so_far[-(WINDOW + 1)]

#     print("\nSTOPPING CHECK (only a Pragmatic method):")
#     print(f"  Window: {WINDOW} iterations")
#     print(f"  Minimum gain required: {MIN_GAIN:.3f}")
#     print(f"  Best PR-AUC {WINDOW} iters ago: {best_so_far[-(WINDOW + 1)]:.4f}")
#     print(f"  Best PR-AUC now:          {best_so_far[-1]:.4f}")
#     print(f"  Gain over last {WINDOW}:  {gain_last_window:.4f}")

#     if gain_last_window >= MIN_GAIN:
#         print("  ✅ Continue: improvement over the last window meets/exceeds the threshold.")
#     else:
#         print("  🟡 Plateau signal: improvement over the last window is below the threshold (consider stopping).")


## 4. Generalization

### 4.1 Summarize tuned CV performance (best trial) across all metrics

In [ ]:
# Purpose: Extract and report the cross-validated performance of the selected best Bayesian trial.
# Justification: This provides transparency into what the optimizer actually optimized (PR-AUC),
# allows comparison between baseline CV, tuned CV, and final test performance, and helps verify
# that observed test-set behavior reflects generalization rather than overfitting to CV.


results = pd.DataFrame(bayes_search.cv_results_)
best_idx = bayes_search.best_index_

summary = {
    'PR_AUC_mean': results.loc[best_idx, 'mean_test_pr_auc'],
    'PR_AUC_std': results.loc[best_idx, 'std_test_pr_auc'],
    'ROC_AUC_mean': results.loc[best_idx, 'mean_test_roc_auc'],
    'ROC_AUC_std': results.loc[best_idx, 'std_test_roc_auc'],
    'Accuracy_mean': results.loc[best_idx, 'mean_test_accuracy'],
    'F1_mean': results.loc[best_idx, 'mean_test_f1'],
    'Recall_mean': results.loc[best_idx, 'mean_test_recall'],
    'Precision_mean': results.loc[best_idx, 'mean_test_precision']
}

print("\nBest tuned XGBoost CV performance (selected by PR-AUC):")
for k, v in summary.items():
    print(f"  {k}: {v:.4f}")


Higher mean + lower standard deviation:  Indicates a genuine improvement: the model performs better and more consistently across folds.

Higher mean + much higher standard deviation: Suggests the gain is likely noisy or unstable, potentially caused by overfitting to specific CV splits.

Similar mean + lower standard deviation: Preferable model choice, as it offers comparable performance with better stability and reliability.

In [ ]:
# Purpose: Compare Baseline vs Tuned XGBoost performance (CV means) across all study metrics,
#          using baseline CV results loaded from model_comparison_analysis.pkl and tuned CV
#          results extracted from BayesSearchCV (summary dict).

# ---------------------------------------------------------
# Global plotting style for readability and consistency
# ---------------------------------------------------------
plt.rcParams.update({
    "font.size": 13,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 13,
})

# ---------------------------------------------------------
# 1) Baseline CV means from model-comparison notebook
# ---------------------------------------------------------
baseline_xgb_scores = cv_results["XGBoost"]

baseline_metrics = {
    "PR-AUC": float(np.nanmean(baseline_xgb_scores["test_pr_auc"])),
    "ROC-AUC": float(np.nanmean(baseline_xgb_scores["test_roc_auc"])),
    "Accuracy": float(np.nanmean(baseline_xgb_scores["test_accuracy"])),
    "Recall": float(np.nanmean(baseline_xgb_scores["test_recall"])),
    "Precision": float(np.nanmean(baseline_xgb_scores["test_precision"])),
    "F1": float(np.nanmean(baseline_xgb_scores["test_f1"])),
}

# ---------------------------------------------------------
# 2) Tuned CV means from Bayesian optimization
# ---------------------------------------------------------
tuned_metrics = {
    "PR-AUC": float(summary["PR_AUC_mean"]),
    "ROC-AUC": float(summary["ROC_AUC_mean"]),
    "Accuracy": float(summary["Accuracy_mean"]),
    "Recall": float(summary["Recall_mean"]),
    "Precision": float(summary["Precision_mean"]),
    "F1": float(summary["F1_mean"]),
}

# ---------------------------------------------------------
# 3) Build comparison table
# ---------------------------------------------------------
metrics_order = ["PR-AUC", "ROC-AUC", "Accuracy", "Recall", "Precision", "F1"]

comp_df = pd.DataFrame({
    "Metric": metrics_order,
    "Baseline CV": [baseline_metrics[m] for m in metrics_order],
    "Tuned CV": [tuned_metrics[m] for m in metrics_order],
})

comp_df["Gain (Tuned − Baseline)"] = comp_df["Tuned CV"] - comp_df["Baseline CV"]

print("\nBaseline vs Tuned XGBoost (Cross-Validation Means):\n")
print(comp_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---------------------------------------------------------
# 4) Visualization: Baseline vs Tuned performance
# ---------------------------------------------------------
x = np.arange(len(metrics_order))
width = 0.36

fig, ax = plt.subplots(figsize=(14, 7))

ax.bar(
    x - width / 2,
    comp_df["Baseline CV"],
    width,
    label="Baseline XGBoost",
    alpha=0.85
)

ax.bar(
    x + width / 2,
    comp_df["Tuned CV"],
    width,
    label="Bayesian-Tuned XGBoost",
    alpha=0.85
)

ax.set_title(
    "XGBoost Performance Comparison: Baseline vs Bayesian Optimization\n"
    "(Cross-Validation Means)",
    fontweight="bold",
    pad=16
)

ax.set_ylabel("Score", fontsize=14)
ax.set_xlabel("Performance Matrices", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics_order, fontsize=12)

ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.3)

# ---------------------------------------------------------
# 5) Annotate gains above tuned bars
# ---------------------------------------------------------
for i, gain in enumerate(comp_df["Gain (Tuned − Baseline)"]):
    y = comp_df.loc[i, "Tuned CV"]
    ax.text(
        x[i] + width / 2,
        y + 0.002,
        f"{gain:+.4f}",
        ha="center",
        va="bottom",
        fontsize=12,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()


#### Compare tuned model performance: excluding vs including sdo78 columns

In [ ]:
# Purpose: Run Bayesian hyperparameter optimization on the same XGBoost model but WITH sdo78 columns
#          included, so we can compare CV performance against the model trained without sdo78.

feature_cols_incl = feature_cols + sdo78_cols
X_train_val_incl = train_val_df[feature_cols_incl].copy()

xgb_base_incl = clone(models_to_tune["XGBoost"]).set_params(scale_pos_weight=spw)

bayes_search_incl = BayesSearchCV(
    estimator=xgb_base_incl,
    search_spaces=xgb_search_space,
    n_iter=N_ITER,
    scoring=scoring,
    refit='pr_auc',
    cv=cv_strategy,
    n_jobs=-1,
    random_state=42,
    verbose=1,
    return_train_score=True
)

bayes_search_incl.fit(X_train_val_incl, y_train_val)

print("Bayesian optimization (incl. sdo78 columns) complete.")
print(f"Best PR-AUC (CV mean): {bayes_search_incl.best_score_:.4f}")
print(f"\nFeatures excl. sdo78: {len(feature_cols)}")
print(f"Features incl. sdo78: {len(feature_cols_incl)}")
print(f"sdo78 columns added:  {len(sdo78_cols)}")

In [ ]:
# Purpose: Compare tuned XGBoost CV performance (mean ± std) with and without sdo78 columns,
#          including Wilcoxon signed-rank tests per metric (paired, same CV folds).

from scipy.stats import wilcoxon

results_excl = pd.DataFrame(bayes_search.cv_results_)
results_incl = pd.DataFrame(bayes_search_incl.cv_results_)

best_idx_excl = bayes_search.best_index_
best_idx_incl = bayes_search_incl.best_index_

metrics_order = ["PR-AUC", "ROC-AUC", "Accuracy", "Recall", "Precision", "F1"]
metric_keys = ["pr_auc", "roc_auc", "accuracy", "recall", "precision", "f1"]

n_splits = N_FOLDS * N_REPEATS

# Extract per-fold scores for both models (best trial)
fold_scores_excl = {}
fold_scores_incl = {}
for k in metric_keys:
    fold_scores_excl[k] = np.array([
        results_excl.loc[best_idx_excl, f"split{s}_test_{k}"] for s in range(n_splits)
    ])
    fold_scores_incl[k] = np.array([
        results_incl.loc[best_idx_incl, f"split{s}_test_{k}"] for s in range(n_splits)
    ])

# Wilcoxon signed-rank test per metric (paired: same folds, same random_state)
p_values = []
for k in metric_keys:
    diff = fold_scores_incl[k] - fold_scores_excl[k]
    if np.all(diff == 0):
        p_values.append(1.0)
    else:
        _, p = wilcoxon(fold_scores_excl[k], fold_scores_incl[k])
        p_values.append(p)

def sig_label(p):
    if p < 0.001:  return "***"
    if p < 0.01:   return "**"
    if p < 0.05:   return "*"
    return "ns"

# Means and stds
means_excl = [fold_scores_excl[k].mean() for k in metric_keys]
stds_excl  = [fold_scores_excl[k].std()  for k in metric_keys]
means_incl = [fold_scores_incl[k].mean() for k in metric_keys]
stds_incl  = [fold_scores_incl[k].std()  for k in metric_keys]

# Print test results table
print("Wilcoxon signed-rank test (Excl. vs Incl. 100 dagen monitor):\n")
print(f"{'Metric':<12} {'Excl. mean':>11} {'Incl. mean':>11} {'p-value':>10} {'Sig.':>5}")
print("-" * 55)
for i, m in enumerate(metrics_order):
    print(f"{m:<12} {means_excl[i]:>11.4f} {means_incl[i]:>11.4f} {p_values[i]:>10.4f} {sig_label(p_values[i]):>5}")

# --- Visualization ---
x = np.arange(len(metrics_order))
width = 0.36

fig, ax = plt.subplots(figsize=(14, 8))

ax.bar(x - width / 2, means_excl, width, yerr=stds_excl, capsize=5,
       label="Excl. 100 dagen monitor", alpha=0.85, edgecolor="black", linewidth=0.8)
ax.bar(x + width / 2, means_incl, width, yerr=stds_incl, capsize=5,
       label="Incl. 100 dagen monitor", alpha=0.85, edgecolor="black", linewidth=0.8)

ax.set_title(
    "Tuned XGBoost: CV Performance Comparison\n"
    "(Excl. vs Incl. 100 dagen monitor data)",
    fontsize=16, fontweight="bold", pad=14
)
ax.set_ylabel("Score", fontsize=14)
ax.set_xlabel("Performance Metrics", fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(metrics_order, fontsize=12)
ax.set_ylim(0, 1.12)
ax.legend(fontsize=13, loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.3)

# # Annotate bars with mean ± std
# for i in range(len(metrics_order)):
#     ax.text(x[i] - width / 2, means_excl[i] + stds_excl[i] + 0.015,
#             f"{means_excl[i]:.4f}\n±{stds_excl[i]:.4f}",
#             ha="center", va="bottom", fontsize=9, fontweight="bold")
#     ax.text(x[i] + width / 2, means_incl[i] + stds_incl[i] + 0.015,
#             f"{means_incl[i]:.4f}\n±{stds_incl[i]:.4f}",
#             ha="center", va="bottom", fontsize=9, fontweight="bold")

# Annotate significance brackets between each pair of bars
for i, p in enumerate(p_values):
    # Height of the bracket: above the taller bar + its error bar + annotation
    y_max = max(means_excl[i] + stds_excl[i], means_incl[i] + stds_incl[i]) + 0.09
    bracket_y = y_max + 0.005

    # Horizontal bracket
    ax.plot(
        [x[i] - width / 2, x[i] - width / 2, x[i] + width / 2, x[i] + width / 2],
        [bracket_y - 0.005, bracket_y, bracket_y, bracket_y - 0.005],
        color="black", linewidth=1.2
    )

    # Significance label
    label = sig_label(p)
    color = "red" if label != "ns" else "gray"
    ax.text(x[i], bracket_y + 0.004,
            f"{label}\n(p={p:.3f})",
            ha="center", va="bottom", fontsize=9, fontweight="bold", color=color)

plt.tight_layout()
plt.show()

### 4.2 Train final tuned model on full train+val and evaluate on test (including PR-AUC)

In [ ]:
# Purpose: Fit the tuned XGBoost model once on the full Train+Validation set and evaluate it once on the
#          untouched Test set using the same threshold-independent metrics used in model comparison.
#          This final evaluation is necessary to confirm that the CV improvement generalizes to future/unseen cohorts.


# --- 1) Build Train+Val and Test matrices ---
train_val_df = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)

X_train_val = train_val_df[feature_cols].copy()
y_train_val = train_val_df[target_col].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df[target_col].copy()

# --- 2) Compute scale_pos_weight on Train+Val (fixed imbalance handling) ---
neg = (y_train_val == 0).sum()
pos = (y_train_val == 1).sum()
spw = (neg / pos) if pos > 0 else 1.0

# --- 3) Rebuild final XGB model with best params + fixed settings ---
best_params = bayes_search.best_params_  # assumes BayesSearchCV already ran
final_xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    verbosity=0,
    n_jobs=-1,
    scale_pos_weight=spw,
    **best_params
)

# --- 4) Fit on Train+Val ---
final_xgb.fit(X_train_val, y_train_val)

# --- 5) Predict probabilities on Test ---
y_test_proba = final_xgb.predict_proba(X_test)[:, 1]

# --- 6) Primary metrics (threshold-free; consistent with model comparison) ---
pr_auc = average_precision_score(y_test, y_test_proba)
roc_auc = roc_auc_score(y_test, y_test_proba)

print("\nFINAL TEST SET EVALUATION (consistent with model comparison):")
print(f"  PR-AUC (primary):   {pr_auc:.4f}")
print(f"  ROC-AUC (secondary): {roc_auc:.4f}")

#### Conclusion

Bayesian hyperparameter tuning yielded a small but consistent improvement over the baseline XGBoost model.

• Baseline cross-validated PR-AUC: ≈ 0.635
• Tuned cross-validated PR-AUC: 0.639
• Cross-validation gain: +0.004

• Baseline test PR-AUC: 0.6195
• Tuned test PR-AUC: 0.6232
• Test-set gain: +0.0037

• Difference between CV gain and test gain: ≈ 0.0003 (≪ 0.005 threshold)

The presence of improvement on both cross-validation and the held-out test set, 

combined with the negligible gap between CV and test gains, indicates that the improvement reflects genuine generalization

 rather than overfitting to validation data. Given the modest magnitude of the test-set improvement and the convergence
 
  of optimal hyperparameters toward stable regions of the search space, further hyperparameter expansion is unlikely to 
  
  produce meaningful gains. Accordingly, tuning was stopped and focus shifted toward generalization and downstream evaluation considerations.

#### NB
To assess the stability of the tuned XGBoost model, we repeated the cross-validation using 

three independent random seeds (42, 0, and 2026), ensuring that the observed PR-AUC improvements 

were not driven by a particular fold split or random initialization. Below are the results. 

|          Seed | CV PR-AUC (mean ± std) | Test PR-AUC         |
| ------------: | ---------------------- | ------------------- |
|            42 | 0.6382 ± 0.0193        | 0.6234              |
|             0 | 0.6373 ± 0.0189        | 0.6213              |
|          2026 | 0.6381 ± 0.0187        | 0.6238              |


Elbow is 5,7, 6 and for seeds 42, 0 and 2026 respectively.

Across three random seeds, both cross-validated and test PR-AUC values remain highly consistent, 

and Bayesian optimization converges within the first 5–7 iterations. 

This provides strong evidence that the observed performance gains are robust, 

reproducible, and not driven by random initialization or optimizer noise.


## 5. Save tuned model + tuning outputs

In [ ]:
# Purpose: Save the final tuned XGBoost model and the key tuning/evaluation outputs
#          (best hyperparameters, CV PR-AUC, test PR/ROC-AUC, and training metadata)
#          so results are reproducible and can be loaded in any notebook later.


# --- 1) Define save paths ---
current_user = os.getenv("USER")
PROCESSED_DIR = Path(f"/home/{current_user}/local-share/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

model_path = PROCESSED_DIR / "xgb_final_tuned_trainval.pkl"
outputs_path = PROCESSED_DIR / "xgb_tuning_outputs.pkl"

# --- 2) Collect tuning + evaluation outputs ---
tuning_outputs = {
    "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "best_model_name": "XGBoost",
    "best_params": dict(bayes_search.best_params_),          # Bayesian best hyperparameters
    "best_cv_pr_auc": float(bayes_search.best_score_),       # CV mean PR-AUC from BayesSearchCV
    "final_scale_pos_weight": float(spw),                    # fixed imbalance weight used in final fit
    "test_metrics_threshold_free": {                         # final test evaluation (threshold-free)
        "pr_auc": float(pr_auc),
        "roc_auc": float(roc_auc),
    },
    "feature_cols": list(feature_cols),
    "target_col": target_col,
    "train_val_size": int(len(y_train_val)),
    "test_size": int(len(y_test)),
}

# --- 3) Save model + outputs ---
joblib.dump(final_xgb, model_path)
joblib.dump(tuning_outputs, outputs_path)

print("Saved tuned model to:", model_path)
print("Saved tuning outputs to:", outputs_path)
